In [10]:
# Import numpy for handling tables and arrays
import numpy as np
# Import gymnasium for the Mountain Car physics simulation
import gymnasium as gym

In [11]:
# Load the classic Mountain Car environment
env = gym.make("MountainCar-v0")

In [12]:
# Define how many "bins" or "buckets" we want for position and velocity
num_bins = 20
# Create evenly spaced boundaries for the position range [-1.2, 0.6]
pos_bins = np.linspace(-1.2, 0.6, num_bins)
# Create evenly spaced boundaries for the velocity range [-0.07, 0.07]
vel_bins = np.linspace(-0.07, 0.07, num_bins)

In [13]:
# Step 1: Create two separate Q-tables initialized to zero: [pos_bucket, vel_bucket, action]
Q1 = np.zeros((num_bins, num_bins, 3))
Q2 = np.zeros((num_bins, num_bins, 3))

In [14]:
# Hyperparameters
alpha = 0.1 # Learning rate (how fast we update our cheat sheet)
gamma = 0.99 # Discount factor (future value focus)
epsilon = 0.1 # Exploration rate (10% chance of random moves)

In [15]:
# Helper function to convert raw decimal state to integer bucket indices
def get_discretized_state(state):
    # Retrieve raw coordinates
    pos, vel = state

    # Find which bin index the values fall into
    pos_idx = np.digitize(pos, pos_bins) - 1
    vel_idx = np.digitize(vel, vel_bins) - 1

    # Ensure indices stay within array boundaries
    return (
        int(np.clip(pos_idx, 0, num_bins - 1)),
        int(np.clip(vel_idx, 0, num_bins - 1))
    )


# Train the agent over 1000 episodes
for episode in range(1000):

    # Reset the environment
    raw_state, info = env.reset()
    state = get_discretized_state(raw_state)
    done = False

    while not done:

        # Step 2: Choose action using epsilon-greedy strategy
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(
                Q1[state[0], state[1]] + Q2[state[0], state[1]]
            )

        # Execute action
        next_raw_state, reward, terminated, truncated, _ = env.step(action)
        next_state = get_discretized_state(next_raw_state)
        done = terminated or truncated

        # Step 3: Randomly update Q1 or Q2
        if np.random.rand() < 0.5:

            # Best action according to Q1
            best_next_action = np.argmax(Q1[next_state[0], next_state[1]])

            # Evaluate using Q2
            target = reward + gamma * Q2[
                next_state[0], next_state[1], best_next_action
            ]

            # Update Q1
            Q1[state[0], state[1], action] += alpha * (
                target - Q1[state[0], state[1], action]
            )

        else:

            # Best action according to Q2
            best_next_action = np.argmax(Q2[next_state[0], next_state[1]])

            # Evaluate using Q1
            target = reward + gamma * Q1[
                next_state[0], next_state[1], best_next_action
            ]

            # Update Q2
            Q2[state[0], state[1], action] += alpha * (
                target - Q2[state[0], state[1], action]
            )

        # Move to next state
        state = next_state


print("Double Q-Tables trained successfully!")

print("Sample Q-values for starting position [-0.5, 0.0]:")

start_idx = get_discretized_state([-0.5, 0.0])

print("Table 1:", Q1[start_idx[0], start_idx[1]])
print("Table 2:", Q2[start_idx[0], start_idx[1]])

Double Q-Tables trained successfully!
Sample Q-values for starting position [-0.5, 0.0]:
Table 1: [-41.1326832  -41.58164807 -41.15453542]
Table 2: [-41.27775598 -41.26305362 -41.68078063]
